# Perf bench: A/B per perf-flag

Minimal notebook to compare the three runtime-toggleable perf fixes in `train_vit.cu`:

- `PERF_SGEMV_BIAS` — bias_grad via cublasSgemv
- `PERF_ATTN_BATCHED` — attention via cublasSgemmBatched
- `PERF_NARROW_MEMSET` — zero ~1 MB instead of ~350 MB per step

No training, no loss curves. Just per-phase wall time per config.

Kaggle setup: 1× T4 is enough, Internet ON (for apt-get OpenMPI).

## 1. Clone repo, install OpenMPI, find NCCL, compile

In [ ]:
!rm -rf /tmp/hpc && git clone https://github.com/SadreevAmir/hpc_final_project /tmp/hpc && cp -r /tmp/hpc/. .

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import subprocess
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'openmpi-bin', 'libopenmpi-dev'], check=True)
print('OpenMPI installed')

In [ ]:
import os, subprocess

hdr = subprocess.check_output(
    'find /opt/conda /usr/include /usr/local -name nccl.h 2>/dev/null | head -n1',
    shell=True, text=True).strip()
lib = subprocess.check_output(
    'find /opt/conda /usr/lib /usr/local -name "libnccl.so*" 2>/dev/null | head -n1',
    shell=True, text=True).strip()
assert hdr and lib, f'NCCL not found: hdr={hdr!r} lib={lib!r}'
os.environ['NCCL_INCLUDE'] = os.path.dirname(hdr)
os.environ['NCCL_LIB']     = os.path.dirname(lib)
print('nccl.h     :', hdr)
print('libnccl.so :', lib)

In [ ]:
!mkdir -p bin
!nvcc -O2 -std=c++17 -ccbin mpicxx -arch=sm_75 \
    -I$NCCL_INCLUDE -L$NCCL_LIB -Xlinker -rpath=$NCCL_LIB \
    src/train_vit.cu -o bin/train_vit \
    -lcublas -lnccl
!ls -lh bin/train_vit

## 2. Locate MNIST CSV

In [ ]:
import glob, os, numpy as np

known = [
    '/kaggle/input/digit-recognizer/train.csv',
    '/kaggle/input/fashionmnist/fashion-mnist_train.csv',
    '/kaggle/input/fashion-mnist/fashion-mnist_train.csv',
]
globbed = sorted(set(
    glob.glob('/kaggle/input/**/*train*.csv', recursive=True) +
    glob.glob('/kaggle/input/**/*Train*.csv', recursive=True)))

def valid_csv(p):
    try:
        with open(p) as f:
            return len(f.readline().split(',')) == 785
    except Exception:
        return False

CSV = next((c for c in known + globbed if os.path.exists(c) and valid_csv(c)), None)
if CSV is None:
    print('Falling back to torchvision MNIST...')
    from torchvision.datasets import MNIST
    ds = MNIST(root='/kaggle/working/mnist_raw', train=True, download=True)
    labels = ds.targets.numpy().astype(np.int32)
    pixels = ds.data.numpy().reshape(-1, 784).astype(np.int32)
    arr = np.concatenate([labels[:, None], pixels], axis=1)
    header = 'label,' + ','.join(f'pixel{i}' for i in range(784))
    CSV = '/kaggle/working/train.csv'
    np.savetxt(CSV, arr, fmt='%d', delimiter=',', header=header, comments='')
assert os.path.exists(CSV)
os.environ['CSV'] = CSV
print('Using CSV:', CSV)

## 3. Bench helper

`run_config` launches the binary with the given flags, parses the resulting
`training_log.csv`, and returns medians (skipping warm-up) for each timing column.

In [ ]:
import csv, subprocess, time
from pathlib import Path

STEPS = 100        # short bench; first 20% are warm-up and discarded
BATCH = 8          # per-rank; same as your throttling runs
LR    = 0.001      # lr does not affect kernel-level timings
WARMUP_FRAC = 0.2

TIMING_COLS = [
    't_h2d_ms', 't_zero_ms', 't_fwd_ms', 't_bwd_ms', 't_nccl_ms', 't_adam_ms',
    'tf_l0_attn', 'tf_l1_attn',
    'tb_lnf', 'tb_l0_ln1', 'tb_l0_ln2', 'tb_l1_ln1', 'tb_l1_ln2',
    'tb_l0_attn', 'tb_l1_attn',
    'tb_l0_qkv', 'tb_l0_aproj', 'tb_l0_fc1', 'tb_l0_fc2',
    'tb_l1_qkv', 'tb_l1_aproj', 'tb_l1_fc1', 'tb_l1_fc2',
]

def run_config(name, flags, steps=STEPS, batch=BATCH, lr=LR):
    env = os.environ.copy()
    for k, v in flags.items():
        env[k] = str(v)
    log_path = Path(f'log_{name}.csv')
    if log_path.exists():
        log_path.unlink()
    t0 = time.time()
    # Note: train_vit.cu always writes ./training_log.csv; we move it.
    proc = subprocess.run(
        ['./bin/train_vit', env['CSV'], str(steps), str(batch), str(lr)],
        env=env, capture_output=True, text=True)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        print(proc.stdout); print(proc.stderr)
        raise RuntimeError(f'{name} failed (rc={proc.returncode})')
    Path('training_log.csv').rename(log_path)

    rows = []
    with log_path.open() as f:
        for r in csv.DictReader(f):
            rows.append({k: float(r[k]) for k in TIMING_COLS if k in r})
    warm = max(1, int(len(rows) * WARMUP_FRAC))
    rows = rows[warm:]
    medians = {k: float(np.median([r[k] for r in rows])) for k in rows[0]}

    tput = None
    for line in proc.stdout.splitlines():
        if 'img/s global' in line:
            tput = float(line.split('|')[0].split(':')[1].strip().split()[0])
            break
    print(f'  {name}: wall {elapsed:.1f}s, tput {tput} img/s global')
    return {'name': name, 'medians': medians, 'wall_s': elapsed,
            'tput': tput, 'stdout': proc.stdout}

## 4. Run configs

Five runs:
1. **baseline** — all three optimizations OFF (original code).
2. **+sgemv** — only bias_grad fix.
3. **+attn** — only batched attention.
4. **+memset** — only narrow memset.
5. **all_fast** — everything ON.

Single-fix runs let you read off the isolated contribution; **all_fast** shows
what the binary delivers end-to-end.

In [ ]:
FLAGS = ('PERF_SGEMV_BIAS', 'PERF_ATTN_BATCHED', 'PERF_NARROW_MEMSET')
# PERF_LN_BWD_FAST is force-pinned to 0 in env (shelved fix).

def cfg(sg, at, ms):
    return dict(zip(FLAGS, (sg, at, ms)), PERF_LN_BWD_FAST=0)

configs = [
    ('baseline',  cfg(0, 0, 0)),
    ('+sgemv',    cfg(1, 0, 0)),
    ('+attn',     cfg(0, 1, 0)),
    ('+memset',   cfg(0, 0, 1)),
    ('all_fast',  cfg(1, 1, 1)),
]

results = []
for name, flags in configs:
    print(f'>>> {name} {flags}')
    results.append(run_config(name, flags))
print('Done.')


## 5. Comparison table

In [ ]:
import pandas as pd

# Coarse phases (ms/step, median)
coarse_cols = ['t_h2d_ms', 't_zero_ms', 't_fwd_ms', 't_bwd_ms', 't_nccl_ms', 't_adam_ms']
df_coarse = pd.DataFrame(
    [{**{'cfg': r['name']}, **{c: r['medians'].get(c, float('nan')) for c in coarse_cols},
      'step_total_ms': sum(r['medians'].get(c, 0) for c in coarse_cols),
      'tput': r['tput']}
     for r in results])
df_coarse['speedup_vs_baseline'] = df_coarse['step_total_ms'].iloc[0] / df_coarse['step_total_ms']
df_coarse.set_index('cfg')

In [ ]:
# Fine: kernels actually touched by each (active) fix
fine_cols = [
    'tb_l0_fc1', 'tb_l0_qkv', 'tb_l1_fc1', 'tb_l1_qkv',     # fix #2 (bias_grad inside)
    'tf_l0_attn', 'tf_l1_attn', 'tb_l0_attn', 'tb_l1_attn', # fix #3
]
df_fine = pd.DataFrame(
    [{**{'cfg': r['name']}, **{c: r['medians'].get(c, float('nan')) for c in fine_cols}}
     for r in results]).set_index('cfg')
df_fine.round(3)


## 6. One stacked-bar: where the step time goes

In [ ]:
import matplotlib.pyplot as plt

phases = ['t_h2d_ms', 't_zero_ms', 't_fwd_ms', 't_bwd_ms', 't_nccl_ms', 't_adam_ms']
data = {p: [r['medians'].get(p, 0) for r in results] for p in phases}
names = [r['name'] for r in results]

fig, ax = plt.subplots(figsize=(8, 4.5))
bottom = np.zeros(len(names))
for p in phases:
    ax.bar(names, data[p], bottom=bottom, label=p.replace('t_', '').replace('_ms', ''))
    bottom += np.array(data[p])
ax.set_ylabel('ms / step (median)')
ax.set_title(f'Per-step time breakdown — B={BATCH}, steps={STEPS} (warmup discarded)')
ax.legend(loc='upper right', fontsize=8)
for i, total in enumerate(bottom):
    ax.text(i, total + 0.05, f'{total:.2f}', ha='center', fontsize=8)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('perf_bench_stack.png', dpi=120)
plt.show()

## 7. Per-fix isolated impact (ms saved vs baseline)

In [ ]:
base = df_coarse.iloc[0]['step_total_ms']
single_fix = df_coarse[df_coarse['cfg'].isin(['+sgemv', '+attn', '+memset'])].copy()
single_fix['ms_saved'] = base - single_fix['step_total_ms']
single_fix['%_of_baseline'] = single_fix['ms_saved'] / base * 100
all_fast_saved = base - df_coarse[df_coarse['cfg'] == 'all_fast']['step_total_ms'].values[0]
print(f'baseline step:    {base:.3f} ms')
print(f'all_fast step:    {df_coarse[df_coarse.cfg=="all_fast"].step_total_ms.values[0]:.3f} ms')
print(f'total saved:      {all_fast_saved:.3f} ms ({all_fast_saved/base*100:.1f}% of baseline)\n')
print('Isolated per-fix savings (each measured with only that fix enabled):')
single_fix[['cfg', 'step_total_ms', 'ms_saved', '%_of_baseline']].to_string(index=False)